[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/5_Ghost_of_Machine/code/baseline/solution.ipynb)

# Ghost of the Machine — baseline on Google Colab

Run all (GPU runtime recommended). The first cell fetches the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-ghost-of-the-machine) and recreates the contest file layout; every cell after it is the original, untouched baseline.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install huggingface_hub')

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-ghost-of-the-machine", repo_type="dataset"))
def link(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src, dst)
# merged dataset root: split folders + *_answers files, public and private together
DSROOT = Path("/content/dsroot").resolve()
for sub in ("public","private"):
    d = DATA/sub
    if d.is_dir():
        for child in d.iterdir():
            link(child, DSROOT/child.name)
print("dataset root:", DSROOT, "->", sorted(p.name for p in DSROOT.iterdir()))

EVAL_SPLIT = "test_leaderboard_a"
ws = Path.cwd()
# layout exactly as the contest mounted it: train answers INSIDE dataset/train/
link(DSROOT/"train"/"data.jsonl", ws/"dataset"/"train"/"data.jsonl")
link(DSROOT/"train_answers.jsonl", ws/"dataset"/"train"/"answers.jsonl")
link(DSROOT/EVAL_SPLIT/"data.jsonl", ws/"dataset"/"test_public"/"data.jsonl")
print("ready: dataset/ (test_public ->", EVAL_SPLIT + ")")


# Ghost of the Machine — baseline

A trivial reference baseline: estimate one number from `dataset/train/` — the average
position of the boundary as a fraction of the passage length — and predict that
same fraction of the length for every test passage.

It exists as a runnable template for the contract: read `dataset/test_public/data.jsonl`,
write `answers.jsonl` at the repository root. Replace the logic below with
your own method.


In [ ]:
import json, os

TRAIN_DIR = "dataset/train"
TEST_DIR  = "dataset/test_public"
OUTPUT    = "answers.jsonl"

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# ---- "train": average boundary position as a fraction of passage length ----
train_rows = read_jsonl(f"{TRAIN_DIR}/data.jsonl")
train_ans = {r["id"]: r["boundary_char_index"] for r in read_jsonl(f"{TRAIN_DIR}/answers.jsonl")}
fracs = [train_ans[r["id"]] / len(r["text"]) for r in train_rows if len(r["text"]) > 0]
mean_frac = sum(fracs) / len(fracs)
print(f"mean boundary fraction from {len(fracs)} train passages: {mean_frac:.4f}")

# ---- predict: same fraction for every test passage ----
test_rows = read_jsonl(f"{TEST_DIR}/data.jsonl")
preds = {r["id"]: int(mean_frac * len(r["text"])) for r in test_rows}

with open(OUTPUT, "w", encoding="utf-8") as f:
    for r in test_rows:
        f.write(json.dumps({"id": r["id"], "boundary_char_index": preds[r["id"]]}) + "\n")
print(f"wrote {OUTPUT}: {len(preds)} predictions")

# self-score when the dev answers are present (absent in the hidden grading set)
ans_path = f"{TEST_DIR}/answers.jsonl"
if os.path.exists(ans_path):
    import math
    gt = {r["id"]: r["boundary_char_index"] for r in read_jsonl(ans_path)}
    scores = [math.exp(-abs(preds[i] - gt[i]) / 100.0) for i in gt if i in preds]
    print(f"self-score: {sum(scores)/len(scores):.4f}")